# Data Indgestion 

In [1]:
# Document Structure

from langchain_core.documents import Document

In [2]:
doc = Document(
    page_content="this is some kind of a text which is in the file", 
    metadata={
        "source" : "smth resource", 
        "pages" : 1, 
        "auther" : "rivindu", 
        "data_created" : "2023-10-01"

    }
)

In [3]:
doc

Document(metadata={'source': 'smth resource', 'pages': 1, 'auther': 'rivindu', 'data_created': '2023-10-01'}, page_content='this is some kind of a text which is in the file')

In [4]:
from langchain_classic.document_loaders import TextLoader, DirectoryLoader

loader = TextLoader("../data/TextFiles/sample.txt", encoding="utf-8")

d:\RAG-Cookbook\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
document = loader.load()

In [6]:
directory_loader = DirectoryLoader(
    path="../data/TextFiles",
    glob="*.txt",
    loader_cls=TextLoader
)

directory_loader.load()

[Document(metadata={'source': '..\\data\\TextFiles\\sample.txt'}, page_content='Python Programming Language\n\nPython is a high-level, interpreted programming language known for its simplicity and readability. Created by Guido van Rossum in 1991, Python emphasizes code clarity through its use of whitespace and straightforward syntax.\n\nKey Features:\n- Easy to learn with beginner-friendly syntax\n- Dynamically typed, allowing flexible variable declaration\n- Extensive standard library for various tasks\n- Supports multiple programming paradigms: procedural, object-oriented, and functional\n- Cross-platform compatibility (Windows, macOS, Linux)\n- Strong community support and rich ecosystem of third-party packages\n\nCommon Applications:\n- Web development with frameworks like Django and Flask\n- Data science and machine learning using NumPy, Pandas, and TensorFlow\n- Automation and scripting\n- Artificial intelligence and natural language processing\n- Scientific computing\n- Game dev

In [39]:
from langchain_classic.document_loaders import PyPDFLoader, DirectoryLoader
pdf_doc = DirectoryLoader("../data/pdfFiles/", 
                          loader_cls=PyPDFLoader, 
                          glob="*.pdf")
pdf_doc.load()


[Document(metadata={'producer': 'xdvipdfmx (20240305)', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-04-30T11:21:16+00:00', 'author': 'Rivindu Ashinsa', 'title': "Rivindu Ashinsa's CV", 'source': '..\\data\\pdfFiles\\RivinduAshinsaResumeGenAI.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='Rivindu Ashinsa\nComputer Science Undergraduate\nLocation: Western Province, Sri Lanka Email: ashinsa.rivindu@gmail.com Phone: +94 752 691 645\nWebsite: rivindu-ashinsa.github.io LinkedIn: rivindu-ashinsa GitHub: rivindu-ashinsa\nProfessional Summary\nComputer Science undergraduate with hands-on experience in data cleaning, analysis, and statistics. Proﬁcient\nin Python, SQL, and data reporting with proven ability to support data-driven decision-making. Experienced\nin processing and analyzing large datasets, creating actionable insights, and building data pipelines. Strong\nanalytical and problem-solving skills with expertise in statistical analysis and data interpret

### Embedding and VectorStoreDB

In [11]:
import numpy as np 
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import Tuple, Any, Dict, Dict, List
from sklearn.metrics.pairwise import cosine_distances

In [10]:
sent_trans = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
sent_trans.get_embedding_dimension()

# sent_trans.encode(["Hello"])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3359.80it/s]


384

In [15]:
class EmbeddingManager:
    def __init__(self,model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model() 
    
    def _load_model(self):
        try:
            print(f"Loading model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print("Model loaded successfully. Embedding dimension:", self.model.get_embedding_dimension())

        except Exception as e:
            print(e)
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if self.model is None:
            print("Model not loaded.")
            self._load_model()
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print("Embeddings generated with shape:", embeddings.shape)
        return embeddings

In [16]:
## Initialize the Embedding manager
embedding_manager = EmbeddingManager()
embedding_manager

Loading model: sentence-transformers/all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6353.98it/s]


Model loaded successfully. Embedding dimension: 384


### Vector Store

In [34]:
import os

class VectorStore:
    def __init__(self, Collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        self.collection_name = Collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        try:
            # FIX: os.makedirs supports exist_ok=True; os.mkdir does not
            os.makedirs(self.persist_directory, exist_ok=True)
            
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # FIX: Use get_or_create_collection so it doesn't crash on the second run
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name, 
                metadata={"Description": "PDF Document embeddings for RAG"}
            )
            
            print(f"Vector Store initialized successfully. Collection: {self.collection_name}")
            print(f"Existing documents: {self.collection.count()}")
        except Exception as e:
            print(f"Initialization failed: {e}")
            raise  # Good practice to raise so the app doesn't run with an empty client

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):  
        if len(documents) != len(embeddings):
            raise ValueError("The number of documents and embeddings must match.")

        print(f"Adding documents to collection: {self.collection_name}")

        ids = []
        metadatas = []
        document_text = []
        embedding_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Safely build metadata dict
            metadata = dict(doc.metadata) if hasattr(doc, 'metadata') and doc.metadata else {}
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            document_text.append(doc.page_content)
            embedding_list.append(embedding.tolist())
    
        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=document_text,
                embeddings=embedding_list
            )
            print(f"Document added successfully. Total documents: {self.collection.count()}")
        except Exception as e:
            print(f"Failed to add documents: {e}")
            raise

# Usage
vector_store = VectorStore()

Vector Store initialized successfully. Collection: pdf_documents
Existing documents: 0


In [48]:
# chunk_texts = [
#     "This is chunk 1: introduction to document ingestion.",
#     "This is chunk 2: embeddings convert text into vectors.",
#     "This is chunk 3: vector stores help with efficient retrieval."
# ]

# chunks = [
#     Document(
#         page_content=text,
#         metadata={"source": "sample_chunks", "chunk_id": i}
#     )
#     for i, text in enumerate(chunk_texts)
# ]

chunks = pdf_doc.load()
chunks

[Document(metadata={'producer': 'xdvipdfmx (20240305)', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-04-30T11:21:16+00:00', 'author': 'Rivindu Ashinsa', 'title': "Rivindu Ashinsa's CV", 'source': '..\\data\\pdfFiles\\RivinduAshinsaResumeGenAI.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='Rivindu Ashinsa\nComputer Science Undergraduate\nLocation: Western Province, Sri Lanka Email: ashinsa.rivindu@gmail.com Phone: +94 752 691 645\nWebsite: rivindu-ashinsa.github.io LinkedIn: rivindu-ashinsa GitHub: rivindu-ashinsa\nProfessional Summary\nComputer Science undergraduate with hands-on experience in data cleaning, analysis, and statistics. Proﬁcient\nin Python, SQL, and data reporting with proven ability to support data-driven decision-making. Experienced\nin processing and analyzing large datasets, creating actionable insights, and building data pipelines. Strong\nanalytical and problem-solving skills with expertise in statistical analysis and data interpret

In [49]:
texts = [doc.page_content for doc in chunks]
len(texts)

2

In [50]:
embeddings = embedding_manager.generate_embeddings(texts)
len(embeddings)

Batches: 100%|██████████| 1/1 [00:00<00:00,  7.61it/s]

Embeddings generated with shape: (2, 384)


2

In [51]:
vector_store.add_documents(chunks, embeddings)

Adding documents to collection: pdf_documents
Document added successfully. Total documents: 5


### RAG Retrieval pipeline


In [ ]:
class RAGRetriver:
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
    
    def retrieve(self, query: str, k: int = 5, score_threshold: float = 0) -> list[Dict[str, Any]]:
        print(f"Retrieving documents for query: {query}")
        print(f"Top {k} documents with similarity score >= {score_threshold}")

    
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            result = self.vector_store.collection.query(
                query_embeddings=[query_embedding],
                n_results=k,    
            )

            # print(result)
            print(result)
            if result['documents'] and result['documents'][0]:
                documents = result['documents'][0]
                metadatas = result['metadatas'][0]
                distances = result['distances'][0]
                ids = result['ids'][0]


        except Exception as e:
            print(e)


retriever = RAGRetriver(vector_store, embedding_manager)

retriever.retrieve("professional summary?")

Retrieving documents for query: professional summary?
Top 5 documents with similarity score >= 0


Batches: 100%|██████████| 1/1 [00:00<00:00, 37.88it/s]

Embeddings generated with shape: (1, 384)
{'ids': [['doc_1e8b7d6c_0', 'doc_3a15fe5f_1', 'doc_44b345ee_1', 'doc_880d3af2_2', 'doc_4591b71d_0']], 'embeddings': None, 'documents': [['This is chunk 1: introduction to document ingestion.', 'This is chunk 2: embeddings convert text into vectors.', 'Key Achievements\n◦ IntelliHack 5.0 (Organized by IEEE UCSC): T op 10 Finalist(T eam: OutlierRejects)\n◦ DataStorm 6.0 Powered by Octave (Organized by UoC & UoM): T op 10 ﬁnalist (T eam: DeepCell)\n◦ SDGP Project: Developed biomedical pipeline for CognivusLabs (University of Westminster)\n◦ Academic Excellence: Achieved 9 A grades in GCE Ordinary Level Examination\nSkills\nProgramming Languages: Python, R(Beginner), Java, SQL (MySQL, MariaDB), HTML, CSS\nMachine Learning & Deep Learning: T ensorFlow, Keras, scikit-learn, T ransformers(Gen AI),CNN, LSTM,\nLangChain, LangGraph, OpenAI API, PyT orch(Beginner), Hugging Face\nData Analysis & Visualization: Pandas, NumPy , Matplotlib, Seaborn, Power BI 